
# Public Demo Setup / Initialization Notebook

This notebook is **not** the public demo itself. It is the **private preparation notebook** you run once to:

1. point the repo at your Hugging Face model repo
2. export a tiny public MAESTRO demo subset from your private Drive/cache
3. upload the public checkpoint + demo assets to Hugging Face
4. sanity-check that the public demo layer can download and load the checkpoint

Use this notebook whenever you want to refresh the public checkpoint or update the curated sample set.

This version keeps the same workflow, but the exported demo assets now include the original MAESTRO MIDI and cached label rolls required by the exact advanced-decoder demo.


In [1]:
#@title 0. Setup repo access and install helper dependencies
from pathlib import Path
from getpass import getpass
import shutil

REPO_CLONE_DIR = "/content/repo"
PROJECT_DIR = f"{REPO_CLONE_DIR}/piano_amt"

TOKEN = getpass("GitHub token: ")

repo_dir = Path(REPO_CLONE_DIR)

# Remove failed/empty clone if needed
if repo_dir.exists() and not (repo_dir / ".git").exists():
    shutil.rmtree(repo_dir)

if not repo_dir.exists():
    !git clone -q "https://$TOKEN@github.com/Mobinmo83/AMT_FYP.git" "$REPO_CLONE_DIR"
    print(f"Cloned → {REPO_CLONE_DIR}")
else:
    print(f"Repo already present at {repo_dir}")
    !cd "$REPO_CLONE_DIR" && git pull -q
    print(f"Pulled latest → {REPO_CLONE_DIR}")

TOKEN = None

%cd $PROJECT_DIR

!pip -q install -r requirements_demo.txt huggingface_hub

print("Setup complete ✓")

GitHub token: ··········
Cloned → /content/repo
/content/repo/piano_amt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 46.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.7/253.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.1 MB/s eta 0:00:00
Setup complete ✓


In [2]:

#@title 1. Imports
import json
import os
import sys
from pathlib import Path

from huggingface_hub import notebook_login

if REPO_CLONE_DIR not in sys.path:
    sys.path.insert(0, REPO_CLONE_DIR)

from demo.export_demo_assets import export_demo_assets
from demo.upload_to_hf import upload_demo_package
from demo.decoder_presets import get_decoder_preset, list_decoder_modes



## 2. Hugging Face authentication

Run the next cell and log in with a Hugging Face token that has permission to create or update your model repo.


In [3]:

#@title 2. Log in to Hugging Face
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
#@title 3. Private paths and public repo settings (edit these)
from pathlib import Path

HF_REPO_ID = "Mobinmo83/piano-amt-demo"
PUBLIC_PRIVATE = False

CHECKPOINT_PATH = "/content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt"
MAESTRO_ROOT = "/content/drive/MyDrive/piano_amt/maestro-v3.0.0"
CACHE_DIR = "/content/drive/MyDrive/piano_amt/cache"
OUTPUT_DEMO_ASSETS = str(Path(REPO_CLONE_DIR) / "demo_assets")
README_HF_PATH = Path(REPO_CLONE_DIR) / "README_HF.md"

# Add 2–5 short stems from your private MAESTRO test set.
# These stems may match either the MAESTRO audio filename or MIDI filename.
DEMO_STEMS = [
    "MIDI-Unprocessed_Recital13-15_MID--AUDIO_15_R1_2018_wav--1",
    "MIDI-Unprocessed_XP_04_R1_2004_03-05_ORIG_MID--AUDIO_04_R1_2004_06_Track06_wav",
    "MIDI-Unprocessed_14_R1_2008_01-05_ORIG_MID--AUDIO_14_R1_2008_wav--3",
    "MIDI-Unprocessed_Recital16_MID--AUDIO_16_R1_2018_wav--3",
    "MIDI-Unprocessed_02_R1_2009_03-06_ORIG_MID--AUDIO_02_R1_2009_02_R1_2009_04_WAV",
    "MIDI-Unprocessed_10_R2_2008_01-05_ORIG_MID--AUDIO_10_R2_2008_wav--2",
]

checkpoint_path = Path(CHECKPOINT_PATH)
maestro_root = Path(MAESTRO_ROOT)
cache_dir = Path(CACHE_DIR)
output_demo_assets = Path(OUTPUT_DEMO_ASSETS)
manifest_path = Path(REPO_CLONE_DIR) / "demo" / "sample_manifest.json"

print("HF repo:", HF_REPO_ID)
print("Checkpoint:", checkpoint_path)
print("MAESTRO root:", maestro_root)
print("Cache dir:", cache_dir)
print("Demo assets dir:", output_demo_assets)
print("Manifest path:", manifest_path)
print("README path:", README_HF_PATH)
print("Demo stems:", DEMO_STEMS)
print("Available decoder modes:", list_decoder_modes())
print("Default quality config:", get_decoder_preset("quality_m2_m3_m4"))
print("Default efficient config:", get_decoder_preset("efficient_m3_m4"))


HF repo: Mobinmo83/piano-amt-demo
Checkpoint: /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
MAESTRO root: /content/drive/MyDrive/piano_amt/maestro-v3.0.0
Cache dir: /content/drive/MyDrive/piano_amt/cache
Demo assets dir: /content/repo/demo_assets
Manifest path: /content/repo/demo/sample_manifest.json
README path: /content/repo/README_HF.md
Demo stems: ['MIDI-Unprocessed_Recital13-15_MID--AUDIO_15_R1_2018_wav--1', 'MIDI-Unprocessed_XP_04_R1_2004_03-05_ORIG_MID--AUDIO_04_R1_2004_06_Track06_wav', 'MIDI-Unprocessed_14_R1_2008_01-05_ORIG_MID--AUDIO_14_R1_2008_wav--3', 'MIDI-Unprocessed_Recital16_MID--AUDIO_16_R1_2018_wav--3', 'MIDI-Unprocessed_02_R1_2009_03-06_ORIG_MID--AUDIO_02_R1_2009_02_R1_2009_04_WAV', 'MIDI-Unprocessed_10_R2_2008_01-05_ORIG_MID--AUDIO_10_R2_2008_wav--2']
Available decoder modes: ['baseline', 'efficient_m3_m4', 'quality_m2_m3_m4', 'm2_only', 'm3_only', 'm4_only']
Default quality config: AdvancedDecoderConfig(name='adv_m2_m3_m4', label='Quality


## 4. Export a tiny public MAESTRO demo subset

This reads your **private** MAESTRO CSV/cache and writes only a few public demo files into your repo:

- `demo_assets/sample_01.wav`
- `demo_assets/sample_01_labels.npz`
- `demo_assets/sample_01_original.mid`
- ...
- `demo/sample_manifest.json`

The `.npz` label rolls are used by the demo for the same evaluation-ground-truth path as the final advanced evaluation. The original MIDI is included for listening, Visual MIDI display, sustain/velocity visualisation, and download.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
#@title 4. Export demo assets from your private dataset/cache
export_demo_assets(
    maestro_root=maestro_root,
    cache_dir=cache_dir,
    output_dir=output_demo_assets,
    names=DEMO_STEMS,
    manifest_path=manifest_path,
)

print("\nManifest contents:\n")
print(manifest_path.read_text())

manifest = json.loads(manifest_path.read_text())
for sample in manifest.get("samples", []):
    assert (Path(REPO_CLONE_DIR) / sample["audio"]).exists(), sample["audio"]
    assert (Path(REPO_CLONE_DIR) / sample["labels"]).exists(), sample["labels"]
    assert (Path(REPO_CLONE_DIR) / sample["midi"]).exists(), sample["midi"]

print(f"\nExported {len(manifest.get('samples', []))} demo samples with audio + labels + original MIDI ✓")


Wrote manifest → /content/repo/demo/sample_manifest.json

Manifest contents:

{
  "samples": [
    {
      "name": "MAESTRO Test 01 \u2014 MIDI-Unprocessed_Recital13-15_MID--AUDIO_15_R1_2018_wav--1",
      "audio": "demo_assets/sample_01.wav",
      "labels": "demo_assets/sample_01_labels.npz",
      "midi": "demo_assets/sample_01_original.mid",
      "metadata": {
        "source_audio_filename": "2018/MIDI-Unprocessed_Recital13-15_MID--AUDIO_15_R1_2018_wav--1.wav",
        "source_midi_filename": "2018/MIDI-Unprocessed_Recital13-15_MID--AUDIO_15_R1_2018_wav--1.midi",
        "split": "test",
        "canonical_composer": "Alexander Scriabin",
        "canonical_title": "Sonata No. 9, Op. 68, \"Black Mass\"",
        "year": "2018",
        "duration_s_audio": 521.2155625,
        "label_frames": 16288,
        "label_fps": 31.25,
        "midi_note_count": 3633,
        "midi_duration_s": 520.8932291666666,
        "has_sustain_cc64": true,
        "sustain_cc64_count": 10241
      }


## 5. Upload checkpoint + demo assets to Hugging Face

This creates or updates the public model repo and uploads:

- `checkpoints/best.pt`
- `demo_assets/*`, including audio, cached labels and original MIDI files
- `demo/sample_manifest.json`
- optional `README.md` from `README_HF.md`


In [7]:

#@title 5. Upload the public demo package to Hugging Face
upload_demo_package(
    repo_id=HF_REPO_ID,
    checkpoint=checkpoint_path,
    demo_assets=output_demo_assets,
    manifest=manifest_path,
    private=PUBLIC_PRIVATE,
    readme=README_HF_PATH,
)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...un_v2/checkpoints/best.pt:  10%|#         | 31.9MB /  318MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

  ...demo_assets/sample_06.wav:  92%|#########2| 22.7MB / 24.7MB            

  ...demo_assets/sample_04.wav:  32%|###2      | 15.5MB / 48.3MB            

  ...demo_assets/sample_03.wav: 100%|##########| 30.5MB / 30.5MB            

  ...sets/sample_04_labels.npz:  25%|##4       | 7.18kB / 28.8kB            

Uploaded checkpoint + demo assets + manifest to Hugging Face repo: Mobinmo83/piano-amt-demo



## 6. Point the repo demo layer at the public HF repo

You can either:

1. edit `demo/demo_config.py` permanently, or
2. set environment variables inside the public notebook.

The public notebook generated for this repo uses environment variables in a config cell, so you usually do **not** need to hard-edit the Python file each time.

The demo code will read the public checkpoint and sample manifest from this HF repo, then use the exact advanced decoder presets from `demo/decoder_presets.py`.


In [8]:
#@title 6. Optional sanity check: test public checkpoint download/load
os.environ["AMT_DEMO_HF_REPO_ID"] = HF_REPO_ID
os.environ["AMT_DEMO_HF_CHECKPOINT_FILENAME"] = "checkpoints/best.pt"
os.environ["AMT_DEMO_REPO_ROOT"] = REPO_CLONE_DIR

from demo.checkpoint import load_demo_model, format_checkpoint_summary
from demo.sources import load_sample_manifest
from demo.decoder_presets import get_decoder_preset

model, ckpt, ckpt_path = load_demo_model("cpu")
print(format_checkpoint_summary(model, ckpt, ckpt_path))

print("\nExact demo decoder defaults:")
print("Baseline :", get_decoder_preset("baseline"))
print("Quality  :", get_decoder_preset("quality_m2_m3_m4"))
print("Efficient:", get_decoder_preset("efficient_m3_m4"))

manifest = load_sample_manifest(manifest_path)
print(f"\nLocal manifest samples: {len(manifest.get('samples', []))}")
print("Sanity check complete ✓")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


checkpoints/best.pt:   0%|          | 0.00/318M [00:00<?, ?B/s]

Checkpoint: /content/amt_demo/checkpoints/checkpoints/best.pt
Epoch: 990
Val loss: 0.045293346999117925
Best val loss: 0.045293346999117925
Global step: 118800
Trainable params: 26,491,256

Exact demo decoder defaults:
Baseline : AdvancedDecoderConfig(name='adv_baseline', label='Tuned baseline — no post-processing', description='No post-processing. Original onset-gated baseline decoder at the tuned prediction thresholds 0.4/0.4.', decoder_type='baseline', onset_threshold=0.4, frame_threshold=0.4, offset_threshold=0.5, use_onset_conditioned_offset=False, use_frame_smoothing=False, frame_smoothing_kernel=7, frame_smoothing_method='median', min_note_duration_ms=16.0, use_duplicate_removal=False, duplicate_tolerance_sec=0.05, use_chord_grouping=False, chord_tolerance_sec=0.03, chord_snap_to='median', use_adaptive_thresholds=False, adaptive_onset_k=0.5, adaptive_frame_k=0.5, use_pedal_extension=False, pedal_energy_threshold=10.0, pedal_max_extension_sec=2.0)
Quality  : AdvancedDecoderConfig